# Project setup and environment

**Inputs:** config/project.yaml; requirements.txt

**Outputs:** reports/00_environment_report.md; workflow_manifest.tsv

This notebook orchestrates existing functions and does not infer unresolved biology.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_context
ctx = load_context(ROOT)


In [ ]:
import importlib.metadata as md
import platform, shutil, subprocess
from src.io_utils import ensure_directories, now_iso

ensure_directories(ctx.root)
packages = ['cobra', 'pandas', 'scipy', 'lxml', 'openpyxl', 'nbformat', 'nbclient']
versions = {name: (md.version(name) if name in {d.metadata['Name'].lower() for d in md.distributions() if d.metadata.get('Name')} else 'NOT_FOUND') for name in packages}
tools = {name: ('FOUND' if shutil.which(name) else 'NOT_FOUND') for name in ['diamond', 'memote', 'git']}
git_commit = 'NOT_A_GIT_REPOSITORY'
if (ctx.root / '.git').exists():
    git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ctx.root, capture_output=True, text=True).stdout.strip()
lines = ['# Environment report', '', f'- Timestamp: {now_iso()}', f'- Project root: {ctx.root}', f'- Python: {platform.python_version()}', f'- Git commit: {git_commit}', '', '## Python packages']
lines += [f'- {k}: {v}' for k, v in versions.items()]
lines += ['', '## External tools'] + [f'- {k}: {v}' for k, v in tools.items()]
(ctx.root / 'reports' / '00_environment_report.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('ENVIRONMENT CHECK COMPLETE')
print(versions, tools)
